In [2]:
import os
from docling.document_converter import DocumentConverter
import os 
from dotenv import load_dotenv

In [2]:
source = "/Users/karim/Desktop/laws for chatbot/Business-Corporations-Act.pdf"
converter = DocumentConverter()
doc= converter.convert(source)

In [3]:
md_output = doc.document.export_to_markdown()

In [4]:
print(md_output)

## 90/2012 Coll.

## ACT

of 25 January 2012

on Commercial Companies and Cooperatives (Business Corporations Act)

The Parliament passed this Act of the Czech Republic:

## PART ONE

## BUSINESS CORPORATIONS

TITLE I

## Chapter 1

## Common provisions

Section 1 [Recodification]

- (1) Business corporations include commercial companies ('compan ies ') and cooperatives.
- (2) Companies include an unlimited partnership and a limited partnership ('partnership s '), a limited -liability company and a jointstock company ('capital compan ies '), as well as a European Company and a European Economic Interest Grouping.
- (3) Cooperatives include a cooperative and a European Cooperative Society.
- (4)  A  European Company, a European Economic Interest Grouping and a European Cooperative Society shall be governed by the provisions of this Act to the extent permitted by directly applicable legislation of the European Union governing the European Company, the European Economic Interest Grouping 

In [5]:
import re
from collections import OrderedDict
def preprocess_markdown(text):
    new_lines = []
    for line in text.split("\n"):
        parts = re.split(r"(Section\s+\d+)", line, flags=re.IGNORECASE)
        if len(parts) <= 2:
            new_lines.append(line)
        else:
            current = parts[0]
            for i in range(1, len(parts)):
                if re.match(r"Section\s+\d+", parts[i], re.IGNORECASE):
                    if current.strip():
                        new_lines.append(current)
                    current = parts[i]
                else:
                    current += parts[i]
            if current.strip():
                new_lines.append(current)
    return "\n".join(new_lines)

def parse_legal_text_final(text, law_name="Business Corporations Act (90/2012)"):
    text = preprocess_markdown(text)

    section_pattern = re.compile(
        r"^(?:#+\s*)?(?:-\s*)?Section\s+(\d+)\s*(?:\[.*?\])?\s*(.*)",
        re.IGNORECASE
    )
    chapter_pattern = re.compile(r"^(?:#+\s*)?Chapter\s+(\d+)", re.IGNORECASE)
    division_pattern = re.compile(r"^(?:#+\s*)?Division\s+(\d+)", re.IGNORECASE)
    subdivision_pattern = re.compile(r"^(?:#+\s*)?Subdivision\s+(\d+)", re.IGNORECASE)
    skip_patterns = [
        re.compile(r"^(?:#+\s*)?TITLE\s+", re.IGNORECASE),
        re.compile(r"^(?:#+\s*)?PART\s+", re.IGNORECASE),
        re.compile(r"^#{1,6}\s*$"),
        re.compile(r"^(?:#+\s*)?BUSINESS\s+CORPORATIONS", re.IGNORECASE),
        re.compile(r"^(?:#+\s*)?COOPERATIVE\s*$", re.IGNORECASE),
    ]

    current_chapter_id = None
    current_chapter_name = None
    current_division_id = None
    current_division_name = None
    expecting_chapter_name = False
    expecting_division_name = False
    current_section_id = None
    current_text_lines = []
    all_chunks = []

    def clean_md(line):
        return re.sub(r"^#+\s*", "", line).strip()

    def should_skip(line):
        for p in skip_patterns:
            if p.match(line):
                return True
        return False

    def save_chunk():
        if current_section_id is None:
            return
        text_joined = " ".join(current_text_lines).strip()
        if not text_joined:
            return
        all_chunks.append({
            "law_name": law_name,
            "chapter_id": current_chapter_id or "Unknown",
            "chapter_name": current_chapter_name or "Unknown",
            "division_id": current_division_id or "None",
            "division_name": current_division_name or "None",
            "section_id": current_section_id,
            "text": text_joined,
        })

    for line in text.split("\n"):
        stripped = line.strip()
        if not stripped:
            continue

        section_match = section_pattern.match(stripped)
        if section_match:
            save_chunk()
            current_section_id = f"Section {section_match.group(1)}"
            current_text_lines = []
            expecting_chapter_name = False
            expecting_division_name = False
            remaining = section_match.group(2).strip()
            if remaining:
                current_text_lines.append(remaining)
            continue

        chapter_match = chapter_pattern.match(stripped)
        if chapter_match:
            save_chunk()
            current_section_id = None
            current_text_lines = []
            current_chapter_id = f"Chapter {chapter_match.group(1)}"
            current_chapter_name = None
            current_division_id = None
            current_division_name = None
            expecting_chapter_name = True
            expecting_division_name = False
            after = re.sub(r"^(?:#+\s*)?Chapter\s+\d+\s*", "", stripped, flags=re.IGNORECASE).strip()
            if after and not division_pattern.match(after) and not section_pattern.match(after):
                current_chapter_name = clean_md(after)
                expecting_chapter_name = False
            continue

        division_match = division_pattern.match(stripped)
        if division_match:
            current_division_id = f"Division {division_match.group(1)}"
            current_division_name = None
            expecting_division_name = True
            expecting_chapter_name = False
            after = re.sub(r"^(?:#+\s*)?Division\s+\d+\s*", "", stripped, flags=re.IGNORECASE).strip()
            if after:
                current_division_name = clean_md(after)
                expecting_division_name = False
            continue

        if subdivision_pattern.match(stripped):
            continue

        if should_skip(stripped):
            continue

        clean = clean_md(stripped)
        if not clean:
            continue

        if expecting_chapter_name:
            current_chapter_name = clean
            expecting_chapter_name = False
            continue

        if expecting_division_name:
            current_division_name = clean
            expecting_division_name = False
            continue

        if current_section_id is not None:
            current_text_lines.append(stripped)

    save_chunk()

    # CLEAN OCR ARTIFACTS
    for chunk in all_chunks:
        t = chunk["text"]
        t = re.sub(r"\s{2,}", " ", t)
        t = t.replace("compan ies", "companies")
        t = t.replace("partnership s", "partnerships")
        t = t.replace("jointstock", "joint-stock")
        t = t.replace(" -liability", "-liability")
        chunk["text"] = t.strip()
        chunk["chapter_name"] = re.sub(r"^#+\s*", "", chunk["chapter_name"]).strip() or "Unknown"

    # SMART DEDUPLICATION: for each section number, keep the chunk with longest text
    merged = OrderedDict()
    for c in all_chunks:
        sid = c["section_id"]
        if sid not in merged:
            merged[sid] = c.copy()
        else:
            # Append text from duplicate, avoid repeating same text
            existing_text = merged[sid]["text"]
            new_text = c["text"]
            if new_text not in existing_text:
                merged[sid]["text"] = existing_text + " " + new_text

    chunks = sorted(merged.values(), key=lambda c: int(c["section_id"].split()[-1]))
    return chunks


chunks = parse_legal_text_final(md_output)
print(f"Total sections parsed: {len(chunks)}")

all_nums = sorted([int(c["section_id"].split()[-1]) for c in chunks])
missing = [i for i in range(all_nums[0], all_nums[-1] + 1) if i not in all_nums]
null_ch = [c for c in chunks if c["chapter_name"] in (None, "Unknown", "")]
empty_text = [c for c in chunks if len(c["text"].strip()) < 10]

print(f"Section range: {all_nums[0]} — {all_nums[-1]}")
print(f"Missing sections: {len(missing)}")
print(f"Unknown chapter names: {len(null_ch)}")
print(f"Empty/short chunks: {len(empty_text)}")

if missing:
    print(f"\nStill missing: {missing}")

if empty_text:
    print(f"\nShort chunks:")
    for c in empty_text[:5]:
        print(f"  {c['section_id']}: '{c['text']}'")

print(f"\n--- Spot checks ---")
for n in [1, 95, 96, 97, 98, 127, 220, 221, 343, 552, 553, 554, 704, 786]:
    found = [c for c in chunks if c["section_id"] == f"Section {n}"]
    if found:
        c = found[0]
        print(f"Section {n}: {c['chapter_id']} | {c['text'][:70]}...")
    else:
        print(f"Section {n}: MISSING")

Total sections parsed: 783
Section range: 1 — 786
Missing sections: 3
Unknown chapter names: 0
Empty/short chunks: 0

Still missing: [127, 343, 704]

--- Spot checks ---
Section 1: Chapter 1 | - (1) Business corporations include commercial companies ('companies '...
Section 95: Chapter 11 | (1) An unlimited partnership is a company with at least two persons pa...
Section 96: Chapter 11 | The trade name shall include the words 'veřejná obchodní společnost', ...
Section 97: Chapter 11 | (1) Mutual legal relations between the members shall be governed by th...
Section 98: Chapter 11 | The memorandum of association shall also include: (a) the company's tr...
Section 127: MISSING
Section 220: Chapter 5 | (1) Members shall have a priority right to participate in an increase ...
Section 221: Chapter 5 | A member may waive his or her priority right in written form with cert...
Section 343: MISSING
Section 552: Chapter 1 | (1) A cooperative shall be a community of an indefinite number of pers..

In [6]:
print(len(chunks))

783


In [7]:
import chromadb
from sentence_transformers import SentenceTransformer

In [8]:
client = chromadb.PersistentClient(path="./data/law_db")
collection = client.get_or_create_collection(
    name="czech_business_law",
    metadata={"hnsw:space": "cosine"},)

Why `get_or_create` instead of just `create`? Because if you run your notebook cell twice, `create_collection` would crash with "collection already exists." `get_or_create` is idempotent — safe to run repeatedly. The `metadata={"hnsw:space": "cosine"}` tells ChromaDB to use **cosine similarity** for distance calculations, which is the standard for text embeddings.


In [9]:
#Data preparing , embedings 
model = SentenceTransformer("all-MiniLM-L6-v2")
text = [c["text"] for c in chunks]
embedings = model.encode(text) #он что numpy array возращает?
print(embedings.shape)

(783, 384)


In [10]:
type(embedings)

numpy.ndarray

In [11]:
print(chunks)

[{'law_name': 'Business Corporations Act (90/2012)', 'chapter_id': 'Chapter 1', 'chapter_name': 'Common provisions', 'division_id': 'None', 'division_name': 'None', 'section_id': 'Section 1', 'text': "- (1) Business corporations include commercial companies ('companies ') and cooperatives. - (2) Companies include an unlimited partnership and a limited partnership ('partnerships '), a limited-liability company and a joint-stock company ('capital companies '), as well as a European Company and a European Economic Interest Grouping. - (3) Cooperatives include a cooperative and a European Cooperative Society. - (4) A European Company, a European Economic Interest Grouping and a European Cooperative Society shall be governed by the provisions of this Act to the extent permitted by directly applicable legislation of the European Union governing the European Company, the European Economic Interest Grouping or the European Cooperative Society."}, {'law_name': 'Business Corporations Act (90/201

In [12]:
all_ids = []
all_documents = []
all_metadatas = []
all_embeddings = embedings.tolist()   # convert entire numpy array to Python lists at once

for i, chunk in enumerate(chunks):

    # --- ids: must be unique strings ---
    all_ids.append(f"sec_{i}")

    # --- documents: the raw text we want to retrieve ---
    all_documents.append(chunk["text"])

    # --- metadatas: structured fields for filtering and citation ---
    # CRITICAL: ChromaDB crashes on None values, so we replace with "Unknown"
    all_metadatas.append({
        "law_name":      chunk.get("law_name", "Unknown") or "Unknown",
        "chapter_id":    chunk.get("chapter_id", "Unknown") or "Unknown",
        "chapter_name":  chunk.get("chapter_name", "Unknown") or "Unknown",
        "division_id":   chunk.get("division_id", "Unknown") or "Unknown",
        "division_name": chunk.get("division_name", "Unknown") or "Unknown",
        "section_id":    chunk.get("section_id", "Unknown") or "Unknown",
    })

# Step 2: Add in batches of 50
# Why batches? Two reasons:
#   1. Large single inserts can timeout or use too much memory
#   2. You get progress feedback so you know it's working

batch_size = 50

for start in range(0, len(all_ids), batch_size):
    end = start + batch_size

    collection.add(
        ids=all_ids[start:end],
        documents=all_documents[start:end],
        metadatas=all_metadatas[start:end],
        embeddings=all_embeddings[start:end],
    )

    # progress indicator
    done = min(end, len(all_ids))
    print(f"Added {done}/{len(all_ids)}")

print(f"\nDone! Collection now has {collection.count()} items.")

Added 50/783
Added 100/783
Added 150/783
Added 200/783
Added 250/783
Added 300/783
Added 350/783
Added 400/783
Added 450/783
Added 500/783
Added 550/783
Added 600/783
Added 650/783
Added 700/783
Added 750/783
Added 783/783

Done! Collection now has 783 items.


In [6]:
def retrieve(query, top_k=5):
    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
    )

    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "section_id": results["metadatas"][0][i]["section_id"],
            "chapter_id": results["metadatas"][0][i]["chapter_id"],
            "chapter_name": results["metadatas"][0][i]["chapter_name"],
            "division_name": results["metadatas"][0][i].get("division_name", ""),
            "text": results["documents"][0][i],
            "distance": round(results["distances"][0][i], 4),
        })
    return retrieved

In [7]:
from google import genai
load_dotenv()
api = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=api)


def generate(query, context_chunks):
    context = ""
    for c in context_chunks:
        label = f"{c['chapter_id']}, {c['section_id']}"
        if c.get("division_name") and c["division_name"] != "Unknown":
            label += f", {c['division_name']}"
        context += f"\n[{label}]\n{c['text']}\n"

        prompt = f"""You are a legal assistant specializing in Czech business law (Act 90/2012 - Business Corporations Act).

            STRICT RULES:
            1. Answer ONLY based on the legal excerpts provided below
            2. Always cite the specific Section number in your answer
            3. If the answer is NOT found in the excerpts, say: "This question is not covered in the provided legal sections."
            4. Do NOT use any outside legal knowledge
            5. Keep your answer practical and concise

            LEGAL EXCERPTS:
            {context}

            USER QUESTION: {query}"""
        
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    return response.text

In [8]:
def ask(question, top_k=5):
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")

    results = retrieve(question, top_k)

    print(f"\nRetrieved {len(results)} sources:")
    for r in results:
        print(f"  [{r['distance']:.3f}] {r['chapter_id']} / {r['section_id']}")

    answer = generate(question, results)

    print(f"\nAnswer:\n{answer}")
    print(f"{'='*60}")
    return answer

In [9]:
ask("What is the minimum number of members required to form a cooperative, and what specific word must be included in its trade name?")


Question: What is the minimum number of members required to form a cooperative, and what specific word must be included in its trade name?


NameError: name 'model' is not defined

In [17]:
print(f"chunks variable: {len(chunks)}")
print(f"ChromaDB collection: {collection.count()}")

result = collection.get(where={"section_id": "Section 552"})
if result["ids"]:
    print(f"\nSection 552 in ChromaDB:")
    print(f"  {result['documents'][0][:150]}")
else:
    print("\nSection 552: NOT IN CHROMADB")

chunks variable: 783
ChromaDB collection: 783

Section 552 in ChromaDB:
  (1) A cooperative shall be a community of an indefinite number of persons, established for the purpose of mutual support of its members or third parti
